In [8]:
import numpy as np
from sklearn.model_selection import KFold
import pandas as pd

# Utilitaires (identique aux contenus du git dans src de la branche main)

ANTENNA_NAMES = ["041A9F60", "041A9F61", "0F173B12", "0F173B13",
                 "PORT_1", "PORT_2", "PORT_3", "PORT_4"]
ANTENNA_POSITIONS = {
    "041A9F60": (2.8, 2.2, 3.6),
    "041A9F61": (2.7, 3.8, 3.6),
    "PORT_1":   (5.7, 4.0, 3.9),
    "PORT_2":   (8.5, 3.8, 3.9),
    "PORT_3":   (5.6, 1.7, 3.9),
    "PORT_4":   (8.4, 2.1, 3.9),
    "0F173B12": (11.0, 1.7, 4.1),
    "0F173B13": (11.2, 4.0, 4.3),
}



# === Loader ===

META_COLS = ["EPC", "xyz", "x", "y", "z", "polar", "nearestAnt", "nearestD"]

class TxRxDsLoader:
    """
    Loader for ds_Tx_Rx.csv.
    
    Features columns: max__rssi__* and max__Delta_rssi__*
    Distances columns: <antenna>__NONE  (e.g., PORT_1__NONE, 041A9F60__NONE)
    Missing RSSI values replaced with -130 dBm.
    """
    def __init__(self):
        self.feature_cols:  list[str] = []
        self.antenna_names: list[str] = []

    def load(self, path):
        df = pd.read_csv(path)

        dist_cols = [
            c for c in df.columns
            if c not in META_COLS
            and c.endswith("__NONE")
            and not c.startswith("max__")
        ]
        self.antenna_names = [c.replace("__NONE", "") for c in dist_cols]

        feat_cols = [
            c for c in df.columns
            if c not in META_COLS and c not in dist_cols
        ]
        self.feature_cols = feat_cols

        ds = df.copy()
        ds[feat_cols] = ds[feat_cols].fillna(-130)

        df_clean = ds[feat_cols + ["x", "y", "z"] + dist_cols].reset_index(drop=True)

        X           = df_clean[feat_cols].values.astype(np.float32)
        y_xyz       = df_clean[["x", "y", "z"]].values.astype(np.float32)
        y_distances = df_clean[dist_cols].values.astype(np.float32)

        return X, y_xyz, y_distances, df_clean



# === cross validation === 

def cross_validate(X, y, train_fn, predict_fn, eval_fn, aggregate_fn, k=5, random_state=42):
    """
    Perform k-fold cross-validation and return detailed results per fold and aggregated summary.
    
    Args:
        X: Input features
        y: Target values
        train_fn: Function to train a model ((X_train, y_train) -> model)
        predict_fn: Function to make predictions ((model, X_test) -> y_pred))
        eval_fn: Function to evaluate predictions ((y_test, y_pred, test_index) -> dict[str, float])
        aggregate_fn: Function to aggregate fold results into summary statistics
            ((List[Dict[str, float]]) -> Dict[str, Dict[str, float]])
        k: Number of folds
        random_state: Random seed for reproductibility
    Returns:
        Dict with two keys:
        - "folds": List of dictionaries, each containing metrics for a fold.
        - "summary": Aggregated summary of metrics across all folds.
    """
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    
    folds_results = []

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = train_fn(X_train, y_train, train_index) ################################################################ MODIFICATION : train_index ajouté 
        y_pred = predict_fn(model, X_test)

        fold_result = eval_fn(y_test, y_pred, test_index) # Passes test_index to allow eval_fn to access other data
        folds_results.append(fold_result)
        summary_result = aggregate_fn(folds_results)
        
    return {
        "folds": folds_results,
        "summary": summary_result
    }
    


# === eval_fn and aggregate_fn ===

def eval_fn(y_true, y_pred, test_index):
    errs = np.linalg.norm(y_pred - y_true, axis=1)
    return {
        "mae":    mae_3d(errs),
        "3Derror": rmse_3d(errs),
        "rmse":   rmse_xyz(y_true, y_pred),
        "acc_05": threshold_accuracy(errs, 0.5),
        "acc_1":  threshold_accuracy(errs, 1.0),
        "acc_2":  threshold_accuracy(errs, 2.0),
        "acc_3":  threshold_accuracy(errs, 3.0),
    }

def aggregate_fn(folds_results):
    summary = {}
    for key in folds_results[0]:
        vals = np.array([f[key] for f in folds_results])
        summary[key] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    return summary



# === Métriques ===

def mae_3d(errs: np.ndarray) -> float:
    """
    Calculate the Mean Absolute Error (MAE) from an array of errors.

    This function works for:
    - distance errors between predicted and true antenna distances
    - Euclidean position errors in 3D space

    Args:
        errs: Array of errors (absolute distance errors or Euclidean errors)

    Returns:
        Mean error.
    """
    return float(np.mean(errs))

def rmse_3d(errs: np.ndarray) -> float:
    """Calculate the 3D Root Mean Squared Error (RMSE) between true and predicted positions.

    The 3D RMSE is the square root of the average squared Euclidean distances between each pair
    of true and predicted 3D coordinates.

    Args: 
        errs: Euclidean distance errors for each sample
    Returns:
        Root Mean Squared Error of the Euclidean distances between predicted and true 3D positions
    """
    return np.sqrt(np.mean(errs ** 2))

def rmse_xyz(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Per-axis average RMSE over x, y, z coordinates.

    Equivalent to sklearn's sqrt(mean_squared_error(y_true, y_pred)) on (n, 3) arrays,
    which averages the squared errors over all samples AND all 3 axes before taking sqrt.

    This differs from rmse_3d by a factor of 1/√3:
        rmse_xyz = rmse_3d(errs) / √3

    Args:
        y_true: True positions, shape (n, 3)
        y_pred: Predicted positions, shape (n, 3)
    Returns:
        Per-axis RMSE (in meters). Smaller than rmse_3d for the same predictions.
    """
    diff = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.sqrt(np.mean(diff ** 2)))

def threshold_accuracy(errs: np.ndarray, threshold: float) -> float:
    """Calculate the percentage of predictions with error below a given threshold.

    Args:
        errs: Euclidean distance errors for each sample
    Returns:
        Percentage of predictions with error ≤ threshold
    """
    return (errs <= threshold).mean()*100



# === print_summary ===

def print_summary(label, summary):
    m = summary
    print(
        f"{label:<35} "
        f"RMSE={m['rmse']['mean']:.3f}±{m['rmse']['std']:.3f}  "
        f"MAE={m['mae']['mean']:.3f}±{m['mae']['std']:.3f}  "
        f"3Derror={m['3Derror']['mean']:.3f}±{m['3Derror']['std']:.3f}  "
        f"≤0.5m={m['acc_05']['mean']:.1f}%  "
        f"≤1m={m['acc_1']['mean']:.1f}%  "
        f"≤2m={m['acc_2']['mean']:.1f}%  "
        f"≤3m={m['acc_3']['mean']:.1f}%"
    )

In [3]:
# Charger les données

print(f"Loading dataset: {"ds_Tx_Rx.csv"}")
loader = TxRxDsLoader()
X, y_xyz, y_distances, df_clean = loader.load("data/ds_Tx_Rx.csv")

results_cv = {}

Loading dataset: ds_Tx_Rx.csv


In [11]:
# ExtraTrees MultiOutput (baseline)

import time
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor, RegressorChain

_params = dict(n_estimators=200, random_state=0, n_jobs=-1)

def train_et_mo(X_train, y_train, train_index):
    m = MultiOutputRegressor(ExtraTreesRegressor(**_params), n_jobs=-1)
    m.fit(X_train, y_train)
    return m

_t0 = time.time()
results_cv["ET_MO"] = cross_validate(
    X, y_xyz,
    train_fn=train_et_mo,
    predict_fn=lambda m, X: m.predict(X),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("ExtraTrees MultiOutput", results_cv["ET_MO"]["summary"])


Temps d'exécution : 176.6s
ExtraTrees MultiOutput              RMSE=0.717±0.011  MAE=1.099±0.014  3Derror=1.242±0.020  ≤0.5m=10.6%  ≤1m=50.1%  ≤2m=93.4%  ≤3m=99.0%


In [4]:
# Utilitaires spécifiques

import numpy as np
from abc import ABC, abstractmethod
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import RegressorChain

COORD_IDX = {"x": 0, "y": 1, "z": 2}

# === Trilatération ===

def _lsq_initial_estimate(distances, antenna_positions):
    """
    Linear Least Squares (slides 16-19).
    Uses B1 as reference point and forms the linear system Ax ≈ b.
    """
    n = len(distances)
    r = distances
    B = antenna_positions

    # A: (n-1) x 3, b: (n-1) x 1
    A = np.zeros((n - 1, 3))
    b = np.zeros(n - 1)

    for i in range(1, n):
        A[i - 1] = B[i] - B[0]
        d_i1 = np.linalg.norm(B[i] - B[0])
        b[i - 1] = 0.5 * (r[0]**2 - r[i]**2 + d_i1**2)

    # Solve Ax = b via least squares (using QR internally)
    x_rel, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

    # x_rel = (x - x1, y - y1, z - z1), so add back B1
    return x_rel + B[0]


def _gauss_newton_step(theta, distances, antenna_positions):
    """
    One iteration of the Gauss-Newton method (slide 25).
    Returns the updated theta.
    """
    n = len(distances)
    r = distances
    B = antenna_positions

    # f(theta): n x 1, residuals d_i(theta) - r_i
    d = np.linalg.norm(B - theta, axis=1)  # true distances from current estimate
    f = d - r

    # J(theta): n x 3, Jacobian of d_i w.r.t. (x, y, z)
    # partial d_i / partial x_j = (x_j - xi_j) / d_i
    J = (theta - B) / d[:, np.newaxis]

    # theta_{k+1} = theta_k - (J^T J)^{-1} J^T f
    JtJ = J.T @ J
    Jtf = J.T @ f
    delta, _, _, _ = np.linalg.lstsq(JtJ, Jtf, rcond=None)

    return theta - delta


def trilaterate(distances, antenna_positions, max_iter=20, tol=1e-8):
    """
    Estimate target position via trilateration (Hereman, CSM).

    1) LSQ (exact linearization) for initial guess
    2) NLSQ Gauss-Newton iteration to refine

    Parameters
    ----------
    distances : array-like, shape (n,)
        Measured distances from the target to each antenna.
    antenna_positions : array-like, shape (n, 3)
        Known (x, y, z) coordinates of each antenna.
    max_iter : int
        Maximum number of Gauss-Newton iterations.
    tol : float
        Convergence tolerance on ||theta_{k+1} - theta_k||.

    Returns
    -------
    theta : np.ndarray, shape (3,)
        Estimated (x, y, z) position of the target.
    """
    distances = np.asarray(distances, dtype=float)
    antenna_positions = np.asarray(antenna_positions, dtype=float)

    # Step 1: LSQ initial estimate
    theta = _lsq_initial_estimate(distances, antenna_positions)

    # Step 1b: If antennas are (near-)coplanar, LSQ cannot determine the
    # out-of-plane coordinate.  Estimate it from distance residuals:
    #   r_i^2 = d_xy_i^2 + (z - z_i)^2
    # where d_xy_i is the in-plane distance from the LSQ solution to antenna i.
    # With z_i ~ z_mean for all i:  z_offset^2 ≈ median(r_i^2 - d_xy_i^2)
    z_spread = np.ptp(antenna_positions[:, 2])  # max - min
    if z_spread < 0.1:  # near-coplanar threshold (meters)
        z_plane = antenna_positions[:, 2].mean()
        d_xy_sq = np.sum((theta[:2] - antenna_positions[:, :2])**2, axis=1)
        z_offset_sq = np.median(distances**2 - d_xy_sq)
        if z_offset_sq > 0:
            z_offset = np.sqrt(z_offset_sq)
            # Try both sides; pick the one closer to any prior z info,
            # defaulting to below the antenna plane (target_z < antenna_z).
            z_below = z_plane - z_offset
            z_above = z_plane + z_offset
            if abs(theta[2] - z_plane) > 0.01:
                # LSQ gave some z hint (slightly non-coplanar) — trust its side
                theta[2] = z_below if theta[2] < z_plane else z_above
            else:
                # No z info at all — use prior: target is below antennas
                theta[2] = z_below

    # Step 2: Gauss-Newton refinement
    for _ in range(max_iter):
        theta_new = _gauss_newton_step(theta, distances, antenna_positions)
        if np.linalg.norm(theta_new - theta) < tol:
            break
        theta = theta_new

    return theta


# === Interfaces ===

class Stage(ABC):
    """Un stage prédit une coordonnée et peut consommer les précédentes."""

    def __init__(self, target: str):
        """
        target : coordonnée prédite par ce stage ("x", "y" ou "z")
        """
        assert target in COORD_IDX, f"target must be 'x', 'y' or 'z', got {target!r}"
        self.target = target
        self.idx    = COORD_IDX[target]

    @abstractmethod
    def fit(self, X_feat: np.ndarray, X_prev: np.ndarray, y_coord: np.ndarray):
        """
        X_feat  : features RSSI  (n, f)
        X_prev  : coordonnées déjà prédites à l'entraînement (n, k), peut être vide (n, 0)
        y_coord : vraie valeur de la coordonnée cible (n,)
        """

    @abstractmethod
    def predict(self, X_feat: np.ndarray, X_prev: np.ndarray) -> np.ndarray:
        """Renvoie les prédictions de la coordonnée cible (n,)."""

# === Implémentations ===

class TrilaterationStage(Stage):
    """
    Utilise la trilatération pure (distances estimées 6> coordonnée).
    Requiert un modèle de distances entraîné séparément.
    """

    def __init__(self, target: str, dist_model, antenna_positions: np.ndarray):
        """
        dist_model        : modèle déjà entraîné qui prédit les distances aux antennes
        antenna_positions : tableau (n_ant, 3)
        """
        super().__init__(target)
        self.dist_model        = dist_model
        self.antenna_positions = np.asarray(antenna_positions)

    def fit(self, X_feat, X_prev, y_coord):
        pass  # le dist_model est supposé déjà entraîné

    def predict(self, X_feat, X_prev) -> np.ndarray:
        dist_preds = self.dist_model.predict(X_feat)   # (n, n_ant)
        out = np.array([
            trilaterate(d, self.antenna_positions)[self.idx]
            for d in dist_preds
        ])
        return out

class ChainStage(Stage):
    """
    RegressorChain sur une seule coordonnée en ajoutant les coordonnées
    déjà prédites comme features supplémentaires.
    """

    def __init__(self, target: str, base_estimator=None):
        super().__init__(target)
        if base_estimator is None:
            base_estimator = ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)
        self.base_estimator = base_estimator
        self._model = None

    def _build_X(self, X_feat, X_prev):
        """Concatène features + coordonnées précédentes."""
        return np.hstack([X_feat, X_prev]) if X_prev.shape[1] > 0 else X_feat

    def fit(self, X_feat, X_prev, y_coord):
        X = self._build_X(X_feat, X_prev)
        self._model = ExtraTreesRegressor(
            **self.base_estimator.get_params()
        ).fit(X, y_coord)

    def predict(self, X_feat, X_prev) -> np.ndarray:
        return self._model.predict(self._build_X(X_feat, X_prev))
    

# === Pipeline ===

class HybridPipeline:
    """
    Orchestre une liste de stages ordonnés.
    Chaque stage reçoit (X_feat, coord_prédites_précédemment).
    L'ordre dans la liste détermine l'ordre de prédiction et les dépendances.
    """

    def __init__(self, stages: list[Stage]):
        # Vérifie que chaque coordonnée x/y/z est prédite au plus une fois
        targets = [s.target for s in stages]
        assert len(targets) == len(set(targets)), "Deux stages ne peuvent pas cibler la même coordonnée."
        self.stages = stages

    def _accumulate(self, X_feat: np.ndarray, n: int) -> np.ndarray:
        """
        Parcourt les stages dans l'ordre, accumule les prédictions.
        Renvoie un tableau (n, 3) avec NaN pour les coordonnées non prédites.
        """
        preds = np.full((n, 3), np.nan)
        for stage in self.stages:
            # Coordonnées déjà prédites (colonnes non-NaN)
            predicted_mask = ~np.isnan(preds[0])
            X_prev = preds[:, predicted_mask]   # (n, k)
            preds[:, stage.idx] = stage.predict(X_feat, X_prev)
        return preds

    def fit(self, X_feat: np.ndarray, y_xyz: np.ndarray):
        """
        Entraîne les stages dans l'ordre.
        À l'entraînement on utilise les VRAIES coordonnées précédentes
        (teacher forcing) pour éviter la propagation d'erreurs.
        """
        preds = np.full((len(X_feat), 3), np.nan)
        for stage in self.stages:
            predicted_mask = ~np.isnan(preds[0])
            X_prev = preds[:, predicted_mask]
            stage.fit(X_feat, X_prev, y_xyz[:, stage.idx])
            # On donne les vraies valeurs aux stages suivants (teacher forcing)
            preds[:, stage.idx] = y_xyz[:, stage.idx]

    def predict(self, X_feat: np.ndarray) -> np.ndarray:
        return self._accumulate(X_feat, len(X_feat))


In [13]:
# Approche hybride trilatératation et regressor chain

ant_pos = np.array([ANTENNA_POSITIONS[a] for a in loader.antenna_names])


def train_hybrid(X_train, y_train, train_index):
    # dist_model entraîné uniquement sur les données du fold courant
    dist_model = ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)
    dist_model.fit(X_train, y_distances[train_index])

    stages = [
        TrilaterationStage("x", dist_model, ant_pos),
        ChainStage("y"),
        ChainStage("z"),
    ]
    pipe = HybridPipeline(stages)
    pipe.fit(X_train, y_train)
    return pipe

# stages = [
#     TrilaterationStage("x", dist_model, ant_pos),
#     TrilaterationStage("y", dist_model, ant_pos),
#     ChainStage("z"),
# ]

# stages = [ChainStage("x"), ChainStage("y"), ChainStage("z")]
_t0 = time.time()
results_cv["trilat + chain ET"] = cross_validate(
    X, y_xyz,
    train_fn=train_hybrid,
    predict_fn=lambda m, Xt: m.predict(Xt),
    eval_fn=eval_fn,
    aggregate_fn=aggregate_fn,
)
print(f"Temps d'exécution : {time.time() - _t0:.1f}s")
print_summary("trilat + chain ET", results_cv["trilat + chain ET"]["summary"])

Temps d'exécution : 184.4s
trilat + chain ET                   RMSE=0.728±0.011  MAE=1.113±0.011  3Derror=1.260±0.019  ≤0.5m=11.4%  ≤1m=49.3%  ≤2m=92.8%  ≤3m=99.0%


In [14]:
# TABLEAU RECAPITULATIF

SEP = "=" * 130
print(f"\n\n{SEP}")
print("RECAPITULATIF (cross-validation 5 folds)")
print(SEP)
header = (
    f"{'Approche':<40} "
    f"{'RMSE':>12} "
    f"{'MAE':>12} "
    f"{'3Derror':>12} "
    f"{'≤0.5m':>8} "
    f"{'≤1m':>8} "
    f"{'≤2m':>8} "
    f"{'≤3m':>8}"
)
print(header)
print("-" * 130)

# Trier les résultats par RMSE (moyenne, croissant)
sorted_results = sorted(
    results_cv.items(),
    key=lambda x: x[1]["summary"]["rmse"]["mean"]
)

for label, res in sorted_results:
    s = res["summary"]
    print(
        f"{label:<40} "
        f"{s['rmse']['mean']:>6.3f}±{s['rmse']['std']:.3f}  "
        f"{s['mae']['mean']:>6.3f}±{s['mae']['std']:.3f}  "
        f"{s['3Derror']['mean']:>6.3f}±{s['3Derror']['std']:.3f}  "
        f"{s['acc_05']['mean']:>5.1f}%  "
        f"{s['acc_1']['mean']:>5.1f}%  "
        f"{s['acc_2']['mean']:>5.1f}%  "
        f"{s['acc_3']['mean']:>5.1f}%"
    )
print(SEP)



RECAPITULATIF (cross-validation 5 folds)
Approche                                         RMSE          MAE      3Derror    ≤0.5m      ≤1m      ≤2m      ≤3m
----------------------------------------------------------------------------------------------------------------------------------
ET_MO                                     0.717±0.011   1.099±0.014   1.242±0.020   10.6%   50.1%   93.4%   99.0%
trilat + chain ET                         0.728±0.011   1.113±0.011   1.260±0.019   11.4%   49.3%   92.8%   99.0%
